# Gemma4EC: predict EC numbers

Reaction SMILES in, EC number (class.subclass.sub-subclass) out.

Requirements: the `gemma4ec` environment, access to [TxGemma](https://huggingface.co/google/txgemma-9b-predict) on
Hugging Face (`huggingface-cli login` once), and a GPU with ≥ 24 GB for the 9B model (≥ 8 GB for 2B).
Run the cells in order from the repository root or from `notebooks/`.

In [ ]:
import os, subprocess, glob
from pathlib import Path
import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(REPO)

ADAPTER = "PlanesLab/Gemma4EC-9B-Predict"      # or PlanesLab/Gemma4EC-2B-Predict
BASE    = "google/txgemma-9b-predict"          # or google/txgemma-2b-predict (must match the adapter)

## 1. Download the adapter (once)

In [ ]:
from huggingface_hub import snapshot_download

adapter_dir = REPO / "models" / ADAPTER.split("/")[-1]
snapshot_download(ADAPTER, local_dir=adapter_dir)
print(adapter_dir)

## 2. Reactions

A CSV with a `rxn_smiles` column. Replace the examples with your own file.

In [ ]:
reactions = pd.DataFrame({"rxn_smiles": [
    "CCO.O=O>>CC=O.OO",
    "OC(=O)CCC(=O)C(=O)O>>OC(=O)CCC(N)C(=O)O",
]})
reactions.to_csv("reactions.csv", index=False)
# reactions = pd.read_csv("my_reactions.csv")

## 3. Preprocess the SMILES (recommended)

Gemma4EC was trained on ECReact, which writes each molecule in one form and keeps one main product per reaction.
`preprocess_smiles.py` rewrites your reactions the same way; predictions are more reliable on this form.

In [ ]:
subprocess.run(["python", "preprocess_smiles.py", "reactions.csv", "reactions_prep.csv"], check=True)
pd.read_csv("reactions_prep.csv")[["rxn_smiles_original", "rxn_smiles"]]

## 4. Predict

In [ ]:
subprocess.run(["python", "predict.py",
                "--raw_data_file", "reactions_prep.csv",
                "--model_checkpoint_folder", str(adapter_dir),
                "--model_id", BASE,
                "--eval_batch_size", "8"], check=True)

## 5. Results

`pred_raw` is the predicted EC number; `smiles` the reaction it answers.

In [ ]:
tsv = sorted(glob.glob("outputs/results/predictions_pred.*.tsv"), key=os.path.getmtime)[-1]
pd.read_csv(tsv, sep="\t")[["smiles", "pred_raw"]]